# RT-MBAS — Exploratory Data Analysis
Run `python -m app.main` first to collect data, then execute these cells.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from analysis.plots import *
from app.config import DATASET_CSV

sns.set_theme(style='darkgrid')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

df = load_df(DATASET_CSV)
print(f'Rows: {len(df)} | Sessions: {df["session_id"].nunique() if "session_id" in df.columns else "N/A"}')
df.head()

In [ ]:
# Label distribution
if 'label' in df.columns:
    ax = df['label'].value_counts().plot.bar(color=['#2ecc71','#e67e22','#e74c3c'], edgecolor='white')
    ax.set_title('Label Distribution')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
    print(df['label'].value_counts(normalize=True).mul(100).round(1).to_string())

In [ ]:
# Time series of core features
fig = plot_time_series(df, features=['eye_aspect_ratio','hand_velocity','motion_entropy','head_tilt_angle'])
plt.show()

In [ ]:
# Correlation heatmap
fig = plot_correlation_heatmap(df)
plt.show()

In [ ]:
# Distributions per label
fig = plot_distributions(df)
plt.show()

In [ ]:
# Q1: Which features correlate with 'high activity'?
if 'label' in df.columns:
    df['is_active'] = (df['label'] == 'Focused').astype(int)
    corr_with_active = df[FEATURE_COLS + ['is_active']].corr()['is_active'].drop('is_active').sort_values(key=abs, ascending=False)
    print('Correlation with Focused state:')
    print(corr_with_active.to_string())

In [ ]:
# Q2: Do facial and hand signals correlate?
facial = ['eye_aspect_ratio','mouth_open_ratio','head_tilt_angle','face_motion_delta']
hand   = ['hand_velocity','hand_acceleration','gesture_activity_score','idle_time_ratio']
cross = df[facial + hand].corr().loc[facial, hand]
fig, ax = plt.subplots(figsize=(8,5))
sns.heatmap(cross, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Cross-modal Correlation: Face vs Hand Features')
plt.tight_layout()
plt.show()

In [ ]:
# Q3: Are there behavioral patterns over time? (rolling average)
df_ts = df.set_index('datetime').sort_index()
fig, axes = plt.subplots(2, 1, figsize=(14,6), sharex=True)
df_ts['eye_aspect_ratio'].rolling('30s').mean().plot(ax=axes[0], label='EAR 30s avg', color='#3498db')
df_ts['hand_velocity'].rolling('30s').mean().plot(ax=axes[1], label='Hand vel 30s avg', color='#e74c3c')
for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
fig.suptitle('Behavioral Patterns Over Time (30s Rolling Average)')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: Behavioral clustering (KMeans)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

X = df[FEATURE_COLS].fillna(0)
X_scaled = StandardScaler().fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, c, title in zip(axes, [clusters, df.get('label', clusters)], ['KMeans Clusters', 'True Labels']):
    scatter = ax.scatter(X_2d[:,0], X_2d[:,1], c=pd.factorize(c)[0], cmap='Set1', s=5, alpha=0.6)
    ax.set_title(title); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle('PCA Projection (2D)')
plt.tight_layout()
plt.show()